In [ ]:
using NBInclude, LinearAlgebra
using SparseArrays
using Nemo
using Serialization
using MAT

In [ ]:
@nbinclude("LRP.ipynb")  #load the solution in LRP form in V(4,4,4|48), which is found by Dumas, Pernet and Sedoglavic.

In [ ]:
L0 = hcat(([LRP[i][1] LRP[i][2] LRP[i][3]] for i in 1:48)...)

In [ ]:
L = reshape(L0, 2304, 1) # transform the solution into a vector form

In [ ]:
[LRP[2][1] LRP[2][2] LRP[2][3]] # verify

In [ ]:
L[961:976]  == vec(LRP[21][1])   # verify

In [ ]:
#The code generate the Jacobian of a solution

function Jacobrent(x, m::Integer, n::Integer, p::Integer, r::Integer)
    A = m * n
    B = n * p
    C = p * m
    s = A + B + C

    X = reshape(x, s, r)

    J = Matrix{eltype(x)}(undef, A * B * C, r * s)

    IA = Matrix{eltype(x)}(I, A, A)
    IB = Matrix{eltype(x)}(I, B, B)
    IC = Matrix{eltype(x)}(I, C, C)

    for t in 1:r
        a = reshape(X[1:A, t], A, 1)
        b = reshape(X[A+1:A+B, t], B, 1)
        c = reshape(X[A+B+1:A+B+C, t], C, 1)

        block = hcat(
            kron(kron(IA, b), c),
            kron(kron(a, IB), c),
            kron(kron(a, b), IC)
        )

        cols = ((t - 1) * s + 1):(t * s)
        J[:, cols] = block
    end

    return J
end

In [ ]:
Js=Jacobrent(L,4,4,4,48)  # the jacobian J(s)

In [ ]:
Jsq = sparse(Rational{BigInt}.(Js)) #transform J(s) in to sparse rational format

In [ ]:
M = matrix(QQ, Matrix(Jsq)) #transform J(s) into "Nemo" type matrix

In [ ]:
dimnull, Ns = nullspace(M)   #compute the Nullspace basis matrix of J(s), which is N(s) defined in the paper

In [ ]:
println("nullity = ", dimnull)

In [ ]:
Ns # The nullspace basis matrix N(s)

In [ ]:
rank(Ns)

In [ ]:
size(Ns)

In [ ]:
Verifys=M*Ns  # This is the matrx Js*Ns, that should be zero

In [ ]:
rank(Verifys)  #check Verifys=Js*Ns is a zero matrix

In [ ]:
serialize("Js.jls", Js)    #save the Jacobian J(s)

In [ ]:
#The code can transfrom a Nemo rational type to Julia rational type

function nemoqq_to_rational(x)
    return parse(BigInt, string(numerator(x))) // parse(BigInt, string(denominator(x)))
end

In [ ]:
m = number_of_rows(Ns)

In [ ]:
n = number_of_columns(Ns)

In [ ]:
Ns_rat = [nemoqq_to_rational(Ns[i, j]) for i in 1:m, j in 1:n] # transform N(s) into rational type

In [ ]:
serialize("Ns.jls", (dimnull = Int(dimnull),  nrows = m,  ncols = n, Ns_rat = Ns_rat)) #save N(s)

In [ ]:
# optional transform the J(s) and N(s) into matlab type data.

In [ ]:
Js_float = Float64.(Js)

In [ ]:
matwrite("Js.mat", Dict("Js" => Js_float))

In [ ]:
Ns_array = [Float64(Ns[i, j]) for i in 1:nrows(Ns), j in 1:ncols(Ns)]

In [ ]:
matwrite("Ns.mat", Dict("Ns" => Ns_array))